# Tracee (tracee-events.json) Sanitization

This notebook sanitizes **Tracee** JSONL logs while preserving security semantics.

## Policy (consistent with the other sanitizers)
- **Usernames:** pseudonymize only `newt` (case-insensitive). Leave all other names unchanged.
- **FQDNs:** pseudonymize **Azure / Azure-hosted** FQDNs only (suffix allowlist). Keep non-Azure domains intact.
- **Hostnames:** optionally pseudonymize `hostName` (environment-specific).
- **Noise lines:** drop Tracee preface `warn` records (e.g., the first two lines in your sample).
- **Everything else** (event types, timestamps, ports, protocols, numeric ids, ordering) stays unchanged.


In [ ]:
import json
import hashlib
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, Tuple


In [ ]:
# ------------------------
# Configuration
# ------------------------
BASE_DIR = Path.cwd().parent.resolve().joinpath("data")

# Input files (JSON Lines)
FILES = {
    "tracee": "tracee-events.json",
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][table] -> list[Path]  (can be 1 or many)
    We keep lists to support multiple occurrences per scenario if they exist.
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for table, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][table].append(p)
    return inputs

INPUTS = collect_inputs(BASE_DIR, FILES)

# Quick sanity print: how many matches per scenario/table
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")

OUT_DIR = BASE_DIR.parent.resolve().joinpath("sanidata")
MAP_DIR = BASE_DIR.parent.resolve().joinpath("mappings")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

SALT_PATH = MAP_DIR / "salt.txt"  # stored locally; do NOT publish
ONLY_PSEUDONYMIZE_USERS = {"newt"}  # lowercase allowlist

# Tracee hostName is environment-specific.
PSEUDONYMIZE_TRACEE_HOST = True

# Azure/Azure-hosted FQDN suffix allowlist (suffix match == wildcard *.suffix)
AZURE_FQDN_SUFFIXES = (
    "trafficmanager.net",
    "azure.com",
    "azure.net",
    "azureedge.net",
    "cloudapp.azure.com",
    "azurewebsites.net",
    "windows.net",
    "database.windows.net",
    "redis.cache.windows.net",
    "servicebus.windows.net",
    "vault.azure.net",
    "azure-api.net",
    "core.windows.net",  # optional; covered by windows.net but kept for clarity
)

def _is_empty(x: Any) -> bool:
    return x is None or (isinstance(x, str) and x.strip() == "")


In [ ]:
# ------------------------
# Stable mapping utilities
# ------------------------
def _load_json(path: Path) -> Dict[str, str]:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

def _save_json(path: Path, obj: Dict[str, str]) -> None:
    path.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def _get_salt() -> bytes:
    if SALT_PATH.exists():
        return SALT_PATH.read_bytes()
    # Create a stable salt if missing (local-only).
    salt = hashlib.sha256(str(Path.cwd()).encode("utf-8")).hexdigest().encode("utf-8")
    SALT_PATH.write_bytes(salt)
    return salt

SALT = _get_salt()

def map_token(value: str, prefix: str, map_name: str, digits: int = 3) -> str:
    """
    Deterministically map a value -> PREFIX_XXX using a salted SHA-256.
    Persists per-category mapping JSON under MAP_DIR.
    """
    if _is_empty(value):
        return value

    v = str(value)
    map_path = MAP_DIR / f"{map_name}.json"
    mapping = _load_json(map_path)

    if v in mapping:
        return mapping[v]

    base = hashlib.sha256(SALT + v.encode("utf-8")).digest()
    modulus = 10 ** digits
    used = set(mapping.values())

    # Collision resolution: re-hash with a counter deterministically
    for ctr in range(0, 10000):
        h = base if ctr == 0 else hashlib.sha256(base + str(ctr).encode("utf-8")).digest()
        num = int.from_bytes(h[:8], "big") % modulus
        token = f"{prefix}_{num:0{digits}d}"
        if token not in used:
            mapping[v] = token
            _save_json(map_path, mapping)
            return token

    raise RuntimeError(f"Too many collisions for {prefix} mapping")


In [ ]:
# ------------------------
# Azure FQDN matching (suffix allowlist)
# ------------------------
def is_azure_fqdn(fqdn: str) -> bool:
    if _is_empty(fqdn):
        return False
    f = str(fqdn).strip().lower().rstrip(".")
    return any(f == sfx or f.endswith("." + sfx) for sfx in AZURE_FQDN_SUFFIXES)

def sanitize_fqdn(fqdn: str) -> str:
    """Only pseudonymize Azure/Azure-hosted FQDNs; keep everything else unchanged."""
    if _is_empty(fqdn):
        return fqdn
    f = str(fqdn).strip().rstrip(".")
    if is_azure_fqdn(f):
        # Add a harmless suffix to avoid looking like a real domain.
        return map_token(f.lower(), "FQDN", "fqdn_map", digits=4) + ".example"
    return f

# Conservative FQDN-like extractor for arbitrary text blobs
FQDN_CANDIDATE_RE = re.compile(
    r"\b([A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?"
    r"(?:\.[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?)+)\b"
)
IPV4_RE = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")

def sanitize_fqdns_in_text(s: str) -> str:
    if _is_empty(s):
        return s
    text = str(s)

    def _repl(m):
        cand = m.group(1)
        # Avoid rewriting plain IPv4 addresses
        if IPV4_RE.match(cand):
            return cand
        return sanitize_fqdn(cand)

    return FQDN_CANDIDATE_RE.sub(_repl, text)


In [ ]:
# ------------------------
# User + hostname sanitizers (newt-only)
# ------------------------
def sanitize_user_token(user: str) -> str:
    """Pseudonymize only allowlisted usernames (case-insensitive)."""
    if _is_empty(user):
        return user
    u = str(user).strip()

    # DOMAIN\USER -> consider only USER part for allowlist
    if "\\" in u:
        dom, name = u.rsplit("\\", 1)
        name_norm = name.strip().lower()
        if name_norm in ONLY_PSEUDONYMIZE_USERS:
            return dom + "\\" + map_token(name_norm, "USER", "user_map", digits=3)
        return u

    name_norm = u.lower()
    if name_norm in ONLY_PSEUDONYMIZE_USERS:
        return map_token(name_norm, "USER", "user_map", digits=3)

    return u

def sanitize_host_token(host: str) -> str:
    if _is_empty(host):
        return host
    h = str(host).strip()
    return map_token(h.lower(), "HOST", "host_map", digits=3)

# Username inside common path patterns
NIX_USER_RE = re.compile(r"(^|\b)(/home/)([^/\s]+)(/)")
WIN_USER_RE = re.compile(r"(^|\b)([A-Za-z]:\\Users\\)([^\\\s]+)(\\)")

def sanitize_user_in_paths(text: str) -> str:
    """Replace only the username segment in common home-directory paths."""
    if _is_empty(text):
        return text
    s = str(text)

    def _nix_repl(m):
        return m.group(1) + m.group(2) + sanitize_user_token(m.group(3)) + m.group(4)

    def _win_repl(m):
        return m.group(1) + m.group(2) + sanitize_user_token(m.group(3)) + m.group(4)

    s = NIX_USER_RE.sub(_nix_repl, s)
    s = WIN_USER_RE.sub(_win_repl, s)
    return s

def sanitize_text(s: str) -> str:
    """Apply string-level sanitization used across log types."""
    if _is_empty(s):
        return s
    x = sanitize_user_in_paths(str(s))
    x = sanitize_fqdns_in_text(x)
    # NOTE: We intentionally do NOT blanket-replace all usernames in free text.
    # Only allowlisted exact tokens ('newt') are replaced where they appear as a standalone field
    # or inside common path patterns.
    return x


In [ ]:
# ------------------------
# Tracee record sanitization
# ------------------------
def should_drop_tracee_record(obj: Dict[str, Any]) -> bool:
    """
    Drop Tracee preface 'warn' lines (noise), e.g.:
      {"level":"warn","ts":...,"msg":"..."}
    """
    return isinstance(obj, dict) and obj.get("level") == "warn" and "timestamp" not in obj

def sanitize_tracee_obj(x: Any) -> Any:
    """Recursively sanitize a Tracee event object."""
    if isinstance(x, dict):
        out = {}
        for k, v in x.items():
            if k == "hostName" and PSEUDONYMIZE_TRACEE_HOST:
                out[k] = sanitize_host_token(v)
            # Conservative: only sanitize allowlisted usernames if a dedicated user-like field appears.
            elif k in {"user", "username", "User", "account", "acct"}:
                out[k] = sanitize_user_token(v)
            else:
                out[k] = sanitize_tracee_obj(v)
        return out

    if isinstance(x, list):
        return [sanitize_tracee_obj(v) for v in x]

    if isinstance(x, str):
        return sanitize_text(x)

    return x

def sanitize_tracee_line(line: str) -> Tuple[bool, str]:
    """Return (keep, sanitized_line). keep=False means drop the line."""
    if line is None:
        return False, ""
    line = line.strip()
    if not line:
        return False, ""

    try:
        obj = json.loads(line)
    except Exception:
        # Non-JSON or truncated line: drop it
        return False, ""

    if should_drop_tracee_record(obj):
        return False, ""

    obj2 = sanitize_tracee_obj(obj)
    return True, json.dumps(obj2, ensure_ascii=False)


In [ ]:
# ------------------------
# Processing driver
# ------------------------
def output_path_for(input_path: Path, scenario: str, out_root: Path) -> Path:
    """Mirror the relative path under out_root/<scenario>/... to avoid name collisions."""
    rel = input_path.relative_to(BASE_DIR)
    out_path = out_root / scenario / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    return out_path

def sanitize_jsonl_file(in_path: Path, out_path: Path, line_sanitizer) -> Dict[str, int]:
    """Stream sanitize a JSONL file. Returns stats."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    kept = dropped = total = 0

    with in_path.open("r", encoding="utf-8", errors="replace") as fin,          out_path.open("w", encoding="utf-8") as fout:
        for raw in fin:
            total += 1
            keep, s = line_sanitizer(raw)
            if not keep:
                dropped += 1
                continue
            fout.write(s + "\n")
            kept += 1

    return {"total": total, "kept": kept, "dropped": dropped}

total = {"total": 0, "kept": 0, "dropped": 0}

for sc in sorted(INPUTS.keys()):
    for in_tracee in INPUTS[sc].get("tracee", []):
        out_tracee = output_path_for(in_tracee, sc, OUT_DIR)
        stats = sanitize_jsonl_file(in_tracee, out_tracee, sanitize_tracee_line)
        for k in total:
            total[k] += stats[k]
        print(f"[{sc}] tracee: {in_tracee} -> {out_tracee} {stats}")

print(f"Done. totals: {total}")
print(f"Outputs root: {OUT_DIR.resolve()}")
print(f"Mappings stored in: {MAP_DIR.resolve()}")


In [ ]:
# ------------------------
# Preview
# ------------------------
def head(path: Path, n: int = 5) -> None:
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.rstrip("\n"))

for sc in sorted(INPUTS.keys()):
    for in_tracee in INPUTS[sc].get("tracee", []):
        out_tracee = output_path_for(in_tracee, sc, OUT_DIR)
        print(f"\n=== tracee-events.sanitized.json ({sc}) ===")
        if out_tracee.exists():
            head(out_tracee, 5)
        else:
            print("(missing)")
